# 🚇 Metro-ASR — Fine-Tuning Guide

Fine-tune Metro-ASR on your own data for domain adaptation.

This notebook covers:
1. Fine-tuning on HuggingFace datasets
2. Fine-tuning on local audio data
3. Training a custom language model
4. Training a custom BPE tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/Metro-ASR/blob/main/examples/fine_tuning.ipynb)

> **Requires GPU** — This notebook uses a T4 GPU for training.

## 📦 Installation

In [ ]:
!pip install metro-asr[train] -q
!git clone https://github.com/MohammedAly22/Metro-ASR.git
%cd Metro-ASR

## 🎯 Option 1: Fine-Tune on HuggingFace Dataset

Edit the configuration variables in `scripts/finetune.py`:

In [ ]:
# Key configuration in scripts/finetune.py:
#
# CONFIG_PATH = "configs/metro_small.yaml"
# TOKENIZER_DIR = "tokenizer_bpe5k_v2"
# PRETRAINED_CHECKPOINT = "checkpoints/metro-small-v2/best_model.pt"
#
# FINETUNE_DATASET = "MohamedRashad/arabic-english-code-switching"
# FINETUNE_LR = 5e-5           # 10-20x lower than pretraining
# FINETUNE_MAX_STEPS = 30000
# FREEZE_ENCODER_STEPS = 3000   # Freeze encoder initially

!CUDA_VISIBLE_DEVICES=0 python scripts/finetune.py

## 📁 Option 2: Fine-Tune on Local Data

Prepare your data as a HuggingFace dataset:

In [ ]:
from datasets import Dataset, Audio

# Your data: list of {audio_path, transcription} pairs
data = [
    {"audio": "path/to/audio1.wav", "text": "أنا رايح الـ meeting"},
    {"audio": "path/to/audio2.wav", "text": "الـ project ده محتاج update"},
    # ... add more samples
]

dataset = Dataset.from_list(data)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
dataset.save_to_disk("my_data/train")

print(f"Dataset saved: {len(dataset)} samples")

In [ ]:
# Then set in scripts/finetune.py:
#   PREPARED_DATA_DIR = "my_data"

!CUDA_VISIBLE_DEVICES=0 python scripts/finetune.py

## 📖 Training a Language Model

Train a KenLM n-gram model for beam search decoding:

In [ ]:
# Edit scripts/train_lm.py:
#   ORDER = 5          # 5-gram
#   OUTPUT_DIR = "lm"  # output directory
#   MAX_SAMPLES = 500000

!python scripts/train_lm.py

## 🔤 Training a BPE Tokenizer

Train a custom BPE tokenizer for Arabic + English:

In [ ]:
# Edit scripts/train_bpe_tokenizer.py:
#   VOCAB_SIZE = 5000
#   OUTPUT_DIR = "my_tokenizer"

!python scripts/train_bpe_tokenizer.py

## 📊 Fine-Tuning Tips

| Parameter | Recommended | Notes |
|-----------|-------------|-------|
| Learning rate | `1e-4` to `5e-5` | 10-20x lower than pretraining |
| Freeze encoder | 3,000 – 10,000 steps | Lets CTC head adapt first |
| Max steps | 20,000 – 50,000 | Depends on dataset size |
| Batch size | 16 – 32 | Same as pretraining |
| SpecAugment | Keep enabled | Helps with small datasets |

### Monitor with WandB

```bash
WANDB_PROJECT=metro-finetune CUDA_VISIBLE_DEVICES=0 python scripts/finetune.py
```

## ✅ Test Your Fine-Tuned Model

In [ ]:
from metro_asr import MetroASREngine

engine = MetroASREngine.from_local(
    config_path="configs/metro_small.yaml",
    checkpoint_path="checkpoints/my-finetune/best_model.pt",
    tokenizer_dir="tokenizer_bpe5k_v2",
    device="cuda",
)

result = engine.transcribe("test_audio.wav")
print(f"Text: {result.text}")
print(f"RTF:  {result.rtf:.4f}")